# RobustDrive exploratory analysis

Inspect recorded data and saved measurements. **PENDING REAL DATA** until you collect simulator recordings. Synthetic output is software verification only. Run from this repository or its notebooks directory after installing the package. This notebook does not train or alter experiments.

In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from robustdrive.data.dataset import load_log

root = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
os.chdir(root)
csv = Path('data/raw/driving_log.csv')
if not csv.exists():
    csv = Path('data/sample/driving_log.csv')
    print('SYNTHETIC SMOKE ONLY — PENDING REAL DATA')
if not csv.exists():
    raise FileNotFoundError('Collect data or run scripts/create_sample_data.py first.')
log = load_log(csv)
print('Data kind:', log.kind)
print('Camera counts:', log.inspection['camera_counts'])
print('Valid label rows:', len(log.records))

In [ ]:
steering = np.array([record.steering for record in log.records])
plt.figure(figsize=(7, 3))
plt.hist(steering, bins=41)
plt.xlabel('Normalized steering (negative = left)')
plt.ylabel('Frames')
plt.title(f'Steering distribution — {log.kind}')
plt.show()
print('Nearly straight fraction:', np.mean(abs(steering) < 0.05))

## Read measurements without inventing results
Choose an existing run below. Confirm its provenance before interpreting any numbers. Clean test error does not prove autonomous driving. Do not tune model choices repeatedly on this test set.

In [ ]:
run = Path('experiments/results/baseline')
if not (run / 'metadata.json').exists():
    run = Path('experiments/results/smoke')
if (run / 'metadata.json').exists():
    metadata = json.loads((run / 'metadata.json').read_text(encoding='utf-8'))
    print(metadata['status'])
    print(pd.read_csv(run / 'history.csv').to_string(index=False))
else:
    print('No trained run. PENDING REAL DATA.')

In [ ]:
selection_path = run / 'selective_prediction.csv'
if selection_path.exists():
    selection = pd.read_csv(selection_path)
    clean = selection[selection.condition == 'clean'].sort_values('actual_coverage')
    plt.figure(figsize=(6, 4))
    plt.plot(100 * clean.actual_coverage, clean.mae, marker='o')
    plt.xlabel('Accepted coverage (%)')
    plt.ylabel('Accepted steering MAE')
    plt.title(metadata['status'], fontsize=9)
    plt.show()
else:
    print('Run evaluate_uncertainty.py on a dropout checkpoint first.')

## Interpretation checklist

- Are the inputs real or synthetic?
- Is the held-out split temporal, and does it contain representative turns?
- Are corruption comparisons paired on the same images and checkpoint?
- Does the coverage curve improve consistently, and which confidently wrong frames remain?
- Are online events actually observed, rather than inferred from offline error?

Document null or contradictory findings as carefully as positive ones.